# DGIM: Counting Ones in a Sliding Window

Wiki reference for [DGIM](https://ml-viz-ruby.vercel.app/wiki/dgim-sliding-window).

**The idea in one sentence.** Summarise the last N bits with power-of-two buckets (at most two of each size) so you can estimate the number of 1s in O(log²N) bits with at most 50% error.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — the bucket structure

Each bucket stores `(size = 2^j, timestamp of its most recent 1)`. On a new 1 we add a size-1 bucket and cascade-merge whenever a third bucket of some size appears; buckets older than `N` are dropped.

In [ ]:
class DGIM:
    def __init__(self, N):
        self.N = N
        self.t = 0
        self.buckets = []   # list of [size, timestamp], newest first
    def update(self, bit):
        self.t += 1
        # drop buckets that have slid out of the last N positions
        self.buckets = [b for b in self.buckets if self.t - b[1] < self.N]
        if bit == 1:
            self.buckets.insert(0, [1, self.t])
            self._merge()
    def _merge(self):
        size = 1
        while True:
            idx = [i for i, b in enumerate(self.buckets) if b[0] == size]
            if len(idx) <= 2:
                break
            # merge the two OLDEST of this size into one of double size
            i2, i1 = idx[-1], idx[-2]
            self.buckets[i1] = [size * 2, self.buckets[i1][1]]   # keep newer timestamp
            del self.buckets[i2]
            size *= 2
    def query(self):
        total, last = 0, 0
        for size, ts in self.buckets:
            if self.t - ts < self.N:
                last = size          # oldest in-window bucket seen so far
                total += size
        return total - last / 2      # count all, but only half the oldest bucket

## 2. Validate against the exact sliding-window count

We run DGIM alongside an exact count over a deque and assert the estimate stays within the promised 50% at every step of a long random stream.

In [ ]:
from collections import deque
N = 1000
dg = DGIM(N)
window = deque(maxlen=N)
rng = np.random.default_rng(0)
max_rel = 0.0
for _ in range(20000):
    bit = int(rng.random() < 0.3)
    dg.update(bit); window.append(bit)
    exact = sum(window)
    if exact > 0:
        max_rel = max(max_rel, abs(dg.query() - exact) / exact)
print(f'exact 1s in window = {sum(window)}   DGIM estimate = {dg.query():.1f}')
print(f'worst relative error over the run = {max_rel:.3%}')
assert max_rel <= 0.5, 'DGIM error is provably at most 50%'
print('DGIM stays within the 50% bound using O(log^2 N) bits ✓')

## 3. Worked trace — reproduce the wiki example

Stream `1 0 1 1 0 1 1 1` with `N = 8`, printing the bucket list after each bit.

In [ ]:
dg2 = DGIM(8)
for bit in [1, 0, 1, 1, 0, 1, 1, 1]:
    dg2.update(bit)
    pretty = ' '.join(f'({s}@{ts})' for s, ts in dg2.buckets)
    print(f'pos {dg2.t}  bit {bit}  ->  {pretty}')
print(f'\nestimate = {dg2.query():.1f}   (true count of 1s = 6)')
assert abs(dg2.query() - 6) / 6 <= 0.5

**What to notice:** the bucket list never holds more than two buckets of any size, and sizes only grow into the past — exactly the two invariants that bound memory to `O(log²N)`. The single halved (oldest) bucket is the only place the estimate can be wrong.

## 4. Gotchas & extensions

- **Error is bounded but two-sided** — halving the oldest bucket can over- or under-count, always within 50%. Keeping `r` buckets per size tightens it to `O(1/r)`.
- The same **exponential-histogram** idea extends to sliding-window sums/variance and underlies [ADWIN](/wiki/adwin).
- **Timestamps are stored mod N** in real implementations to stay `O(log N)` bits; comparisons must handle wraparound.

## 5. Your turn

### Exercise — the DGIM query

Given a list of in-window buckets `[[size, ts], …]` (newest first) and the current time, return the DGIM estimate: **sum every bucket's size, but count only half of the oldest bucket.**

In [ ]:
def dgim_query(buckets, t, N):
    # TODO(you): sum sizes of buckets still in the window; subtract half the OLDEST one
    return ...


In [ ]:
# Checks — run me
# buckets newest-first: sizes 1,1,2,2 all in-window; oldest is size 2
bk = [[1, 8], [1, 7], [2, 6], [2, 3]]
assert dgim_query(bk, t=8, N=8) == (1 + 1 + 2 + 2) - 2 / 2, 'count all, half the oldest'
assert dgim_query([[4, 10]], t=10, N=8) == 4 - 4 / 2, 'single bucket -> half of it'
assert dgim_query([], t=5, N=8) == 0, 'empty window -> 0'
print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def dgim_query(buckets, t, N):
    total, last = 0, 0
    for size, ts in buckets:
        if t - ts < N:
            last = size
            total += size
    return total - last / 2
```

</details>

## 6. Key takeaways

- DGIM summarises a length-`N` window with **power-of-two buckets, ≤2 per size** → `O(log²N)` bits.
- A new 1 triggers **cascading merges**; the query sums buckets plus **half the oldest** → **≤50%** error.
- Back to [Streaming Algorithms & Sketches](https://ml-viz-ruby.vercel.app/courses/streaming-ml/02-streaming-algorithms).